In [ ]:
from pathlib import Path
# Paths relative to this notebook's location (notebooks/ dir)
PROJECT_ROOT = Path().resolve().parent
DRHARD_ROOT = PROJECT_ROOT.parent / "DRhard"

In [14]:
%load_ext autoreload
%autoreload 2

from time import time
import pandas as pd
import numpy as np
import os
from collections import Counter, defaultdict
import pickle
import sys
sys.path.insert(0, str(DRHARD_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Load qrels

In [8]:
# GET qrels
qrel_path=str(PROJECT_ROOT / "data/CAST_qrels/qrels-docs.2019.txt")
qrels_df = pd.read_csv(qrel_path, delimiter=" ", header=None)

# rilevanti
qrels_df = qrels_df[qrels_df[3]>0]

# lista rilevanti
rel_docs_list  = list(qrels_df[2])
len(rel_docs_list)

In [39]:
with open(str(DRHARD_ROOT / "data/passage/relevant_docs_ids.txt"), 'w') as filehandle:
    for index, row in qrels_df.iterrows():
        filehandle.write("{}\t{}\n".format(row[0], row[2]))

# Load embeddings

In [9]:
doc_memmap_path = str(DRHARD_ROOT / "data/passage/evaluate/star/passages.memmap")
docid_memmap_path = str(DRHARD_ROOT / "data/passage/evaluate/star/passages-id.memmap")

In [10]:
doc_embeddings = np.memmap(doc_memmap_path, dtype=np.float32, mode="r")
doc_ids = np.memmap(docid_memmap_path, dtype=np.int32, mode="r")
doc_embeddings = doc_embeddings.reshape(-1, 768)

# Load id maps

In [11]:
# collection id dict
collection_mapping_path = str(DRHARD_ROOT / "data/passage/dataset/CASTcollectionPID2newID.tsv")
collection_df = pd.read_csv(collection_mapping_path, delimiter="\t", header=None)
print(len(collection_df))

38626614


In [12]:
pid2newpid_dict = dict(zip(collection_df[0], collection_df[1])) 
newpid2pid_dict = dict(zip(collection_df[1], collection_df[0])) 

In [15]:
# DRhard docid and qid encoding
preprocess_dir = str(DRHARD_ROOT / "data/passage/preprocess")

pid2offset = pickle.load(open(os.path.join(preprocess_dir, "pid2offset.pickle"), 'rb'))
offset2pid = {v:k for k, v in pid2offset.items()}

# Convert passage ids to star ids

In [17]:
pid2newpid_rel_doc_list = [pid2newpid_dict[x] for x in rel_docs_list]

In [18]:
newpid2starid = [pid2offset[x] for x in pid2newpid_rel_doc_list]

In [19]:
relevant_embeddings = doc_embeddings[newpid2starid]

In [22]:
relevant_embeddings.shape

(8120, 768)

# Save embeddings to file

In [21]:
relevant_embeddings.tofile(str(DRHARD_ROOT / "data/passage/relevant_docs_embeddings.memmap"))

# Check for normalization

In [24]:
relevant_embeddings[0]

In [26]:
np.sqrt(sum([x*x for x in relevant_embeddings[0]]))

24.84152585963421